# Quy-model: Hand Gesture Image Classification
Train a CNN on the gesture image dataset and evaluate it.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
DATA_DIR = Path('../data/raw')
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 3

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Load only folders that contain images
class_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir() and any(d.rglob('*.jpg'))]
print('classes:', [d.name for d in class_dirs])

In [ ]:
# Build a flat list of (path, label)
samples = []
class_names = sorted([d.name for d in class_dirs])
class_to_idx = {c:i for i,c in enumerate(class_names)}
for d in class_dirs:
    for f in d.rglob('*.jpg'):
        samples.append((f, class_to_idx[d.name]))
print('num samples:', len(samples))

In [ ]:
from torch.utils.data import Dataset
class SimpleDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

ds = SimpleDataset(samples, transform)
n_val = int(len(ds)*0.2)
n_train = len(ds)-n_val
train_ds, val_ds = random_split(ds, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print('train:', len(train_ds), 'val:', len(val_ds))

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        total += y.size(0)
    print(f'Epoch {epoch+1}: train_acc={correct/total:.4f} loss={loss_sum/total:.4f}')

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        correct += (out.argmax(1)==y).sum().item()
        total += y.size(0)
print('Val accuracy:', correct/total)

In [ ]:
# Predict on one sample image
sample_path = samples[0][0]
img = Image.open(sample_path).convert('RGB')
x = transform(img).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(x)
    pred = class_names[out.argmax(1).item()]
print('Predicted:', pred, 'for', sample_path.name)
plt.imshow(img); plt.title(pred); plt.axis('off'); plt.show()